In [1]:
# =========================================================
# submit_batches.py
# Run this first. It submits jobs to OpenAI and immediately
# records the batch IDs on HF (NOT on Kaggle's local disk),
# so a Kaggle crash afterwards can't lose track of anything.
# =========================================================
import os
import json
from datasets import load_dataset
from openai import OpenAI
from huggingface_hub import HfApi, hf_hub_download

# =========================================================
# Config
# =========================================================
os.environ.setdefault("OPENAI_API_KEY", "sk_key") # Access key removed for safety
os.environ["HF_TOKEN"] = "hf_token" # Access token removed for safety

OPENAI_MODEL = "gpt-4.1"
MAX_NEW_TOKENS = 768
CHUNK_SIZE = 10  

TEST_DATASET_NAME = "businessrules/dataset_stratified_test"
TRACKING_REPO = "businessrules/GPT4_batch_tracking"  
TRACKING_FILE = "batch_jobs.json"

client = OpenAI()
api = HfApi()

from huggingface_hub import login
login(os.environ["HF_TOKEN"])


def build_prompt(code):
    return f"""### Instruction:
Extract the business rules implemented by the following code.

Constraints:
- Return ONLY the business rules
- Use Markdown bullet points
- DO NOT repeat the instruction
- DO NOT repeat the code
- DO NOT add explanations
- DO NOT wrap output in code blocks
- End when the rules end

### Code:
{code}

### Business Rules:
"""


# =========================================================
# Load existing tracking file from HF (if any), so reruns
# don't resubmit chunks that are already in flight/done.
# =========================================================
def load_tracking():
    try:
        api.create_repo(repo_id=TRACKING_REPO, repo_type="dataset", exist_ok=True)
        path = hf_hub_download(repo_id=TRACKING_REPO, repo_type="dataset", filename=TRACKING_FILE)
        with open(path, "r") as f:
            return json.load(f)
    except Exception:
        return {"submitted_batches": [], "submitted_ids": []}


def save_tracking(tracking):
    with open(TRACKING_FILE, "w") as f:
        json.dump(tracking, f, indent=2)
    api.upload_file(
        path_or_fileobj=TRACKING_FILE,
        path_in_repo=TRACKING_FILE,
        repo_id=TRACKING_REPO,
        repo_type="dataset",
    )


tracking = load_tracking()
already_submitted_ids = set(tracking["submitted_ids"])

test_dataset = load_dataset(TEST_DATASET_NAME, split="test")

SLICE_RANGE = None
if SLICE_RANGE is not None:
    test_dataset = test_dataset.select(SLICE_RANGE)
    print(f"Using a slice of {len(test_dataset)} examples (indices {SLICE_RANGE.start}-{SLICE_RANGE.stop - 1}).")

remaining_examples = [ex for ex in test_dataset if str(ex["id"]) not in already_submitted_ids]
print(f"{len(remaining_examples)} examples not yet submitted (out of {len(test_dataset)}).")


def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


chunks = list(chunk_list(remaining_examples, CHUNK_SIZE))
print(f"Submitting {len(chunks)} chunk(s) of up to {CHUNK_SIZE} examples each.")

for i, chunk in enumerate(chunks, start=1):
    batch_input_path = f"batch_input_{i}.jsonl"

    with open(batch_input_path, "w") as f:
        for example in chunk:
            request_obj = {
                "custom_id": str(example["id"]),
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": OPENAI_MODEL,
                    "messages": [{"role": "user", "content": build_prompt(example["cd"])}],
                    "max_tokens": MAX_NEW_TOKENS,
                    "temperature": 0,
                },
            }
            f.write(json.dumps(request_obj) + "\n")

    with open(batch_input_path, "rb") as f:
        uploaded_file = client.files.create(file=f, purpose="batch")

    batch_job = client.batches.create(
        input_file_id=uploaded_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    print(f"Chunk {i}: submitted batch job {batch_job.id} ({len(chunk)} examples)")

    tracking["submitted_batches"].append({
        "batch_id": batch_job.id,
        "ids": [str(ex["id"]) for ex in chunk],
        "collected": False,
    })
    tracking["submitted_ids"].extend([str(ex["id"]) for ex in chunk])
    save_tracking(tracking)

print("\nAll chunks submitted. You can close Kaggle now — "
      "the jobs run on OpenAI's servers regardless.")
print(f"Batch IDs are safely tracked at https://huggingface.co/datasets/{TRACKING_REPO}")
print("Run collect_results.py anytime later (even in a brand new session) to pull results.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


README.md:   0%|          | 0.00/510 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.85M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/486k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

200 examples not yet submitted (out of 200).
Submitting 20 chunk(s) of up to 10 examples each.
Chunk 1: submitted batch job batch_6a7898d3d9d88190a722e075979390ff (10 examples)
Chunk 2: submitted batch job batch_6a7898d5b28881908faf08874c02b476 (10 examples)
Chunk 3: submitted batch job batch_6a7898d6d2688190bd08afbde82836df (10 examples)
Chunk 4: submitted batch job batch_6a7898d7f4b8819099e9ac367b1641da (10 examples)
Chunk 5: submitted batch job batch_6a7898d91c188190afcc380578ceeaca (10 examples)
Chunk 6: submitted batch job batch_6a7898da0ef081908e8dd9a542bae42c (10 examples)
Chunk 7: submitted batch job batch_6a7898db0ed08190b5558ba027096fea (10 examples)
Chunk 8: submitted batch job batch_6a7898dc17dc8190b257b9085d31b45a (10 examples)
Chunk 9: submitted batch job batch_6a7898dd06e08190bf27a03453b5f01f (10 examples)
Chunk 10: submitted batch job batch_6a7898de3f34819098d1bc465591a67d (10 examples)
Chunk 11: submitted batch job batch_6a7898df95408190a2ef612361fee897 (10 examples)
C